# 📊 Notebook 12 — Analyse Approfondie du Régresseur de Quantités

> **Date** : 2026-08-15  
> **Cas d'étude** : Client `CLT070730` / Produit `25078RA3EABLACK4/128` (REDMI 15C MIDNIGHT BLACK 4/128GB)  
> **Prédiction obtenue** : `quantite_suggeree = 192`  
> **Objectif** : Comprendre d'où vient cette prédiction, si elle est raisonnable, et identifier les limites du régresseur actuel.

---

## 1. Contexte : Rôle du Régresseur dans le Pipeline

Le pipeline de recommandation fonctionne en **deux étapes** :

```
┌─────────────────────┐     ┌─────────────────────┐
│   CLASSIFIEUR       │     │   RÉGRESSEUR         │
│   XGBoost           │ ──▶ │   XGBoost            │
│                     │     │                      │
│ "Ce client va-t-il  │     │ "Si oui, COMBIEN     │
│  acheter ce produit?"│     │  d'unités va-t-il    │
│                     │     │  commander ?"         │
│ Sortie: probabilité │     │ Sortie: quantité     │
│ (0 à 1)             │     │ (nombre entier)      │
└─────────────────────┘     └─────────────────────┘
```

Le régresseur ne s'exécute que sur les produits que le classifieur a identifiés comme "va probablement acheter". Son objectif est de **prédire la quantité** de la prochaine commande.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import warnings
warnings.filterwarnings('ignore')

# Chargement
df = pd.read_csv('../data/processed/training_set.csv')
regressor = joblib.load('../src/models/regressor_lsat.joblib')
with open('../src/models/regressor_lsat_metadata.json', 'r') as f:
    metadata = json.load(f)

print(f"Dataset : {len(df)} lignes")
print(f"Régresseur entraîné le : {metadata['trained_at'][:10]}")
print(f"Nombre de features : {metadata['n_features']}")
print(f"Meilleur arbre (early stopping) : {metadata['best_iteration']}")

---

## 2. Historique RÉEL des Commandes — Client CLT070730

Avant de juger la prédiction du régresseur, regardons ce que ce client a **vraiment commandé** pour ce produit.

In [ ]:
# Charger les commandes brutes
lignes = pd.read_csv('../data/processed/lignes_clean.csv')
cmd = pd.read_csv('../data/processed/commandes_clean.csv')
merged = lignes.merge(cmd, on='code_facture', how='left')

# Filtrer sur le client et le produit exact
orders = merged[
    (merged['code_client'] == 'CLT070730') & 
    (merged['code_article'] == '25078RA3EABLACK4/128')
].sort_values('date_commande').copy()

orders['date_commande'] = pd.to_datetime(orders['date_commande'])

print(f"Produit : REDMI 15C MIDNIGHT BLACK 4/128GB")
print(f"Client  : CLT070730")
print(f"Nombre de commandes : {len(orders)}")
print()

# Tableau des commandes
display_df = orders[['date_commande', 'code_facture', 'quantite']].reset_index(drop=True)
display_df.index = range(1, len(display_df) + 1)
display_df.index.name = '#'
display_df.columns = ['Date', 'Facture', 'Quantité']
display_df

In [ ]:
# Visualisation de l'historique des commandes
fig, ax = plt.subplots(figsize=(14, 5))

dates = orders['date_commande'].values
qtys = orders['quantite'].values

# Barres pour chaque commande
bars = ax.bar(range(len(qtys)), qtys, color='#3b82f6', alpha=0.8, width=0.6, edgecolor='#1e40af')

# Ligne moyenne
avg = np.mean(qtys)
ax.axhline(y=avg, color='#ef4444', linestyle='--', linewidth=2, label=f'Moyenne = {avg:.0f}')

# Ligne prédiction régresseur
ax.axhline(y=192, color='#22c55e', linestyle='-', linewidth=2.5, label='Prédiction régresseur = 192')

# Étiquettes sur les barres
for i, (bar, q) in enumerate(zip(bars, qtys)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8, 
            str(q), ha='center', va='bottom', fontweight='bold', fontsize=10)

# Formatage
ax.set_xticks(range(len(dates)))
date_labels = [pd.Timestamp(d).strftime('%d/%m/%y') for d in dates]
ax.set_xticklabels(date_labels, rotation=45, ha='right')
ax.set_ylabel('Quantité commandée', fontsize=12)
ax.set_title('CLT070730 — Historique des commandes REDMI 15C BLACK 4/128GB', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=11)
ax.set_ylim(0, max(qtys) + 60)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nStatistiques des quantités :")
print(f"  Min     : {min(qtys)}")
print(f"  Max     : {max(qtys)}")
print(f"  Moyenne : {avg:.1f}")
print(f"  Médiane : {np.median(qtys):.0f}")
print(f"  Écart-type : {np.std(qtys):.1f}")
print(f"  Dernière commande : {qtys[-1]} unités (le {date_labels[-1]})")

### 📌 Ce que l'historique nous montre

Les commandes de ce client pour ce produit sont **extrêmement irrégulières** :

| Statistique | Valeur |
|---|---|
| Min | 9 unités |
| Max | 500 unités |
| Moyenne | 145.9 unités |
| Écart-type | ~158 unités |
| **Coefficient de variation** | **108%** ❗ |

Un **écart-type plus grand que la moyenne** (158 > 146) signifie que ce client commande de manière très imprévisible — tantôt 9 unités, tantôt 500.

> **C'est le cas le plus difficile pour un régresseur** : prédire une quantité quand le client lui-même ne suit aucun pattern stable.

---

## 3. Décortiquer la Prédiction du Régresseur : Pourquoi 192 ?

### 3.1 Les features que le régresseur a reçues

In [ ]:
# Simuler exactement le pipeline de recommendation.py
regressor_features = [
    'avg_qty', 'std_qty', 'min_qty', 'max_qty', 'last_qty',
    'frequency', 'recency_days', 'avg_delay_days',
    'current_month_coef', 'avg_seasonal_coef'
]

# Isoler le client avec deduplication (comme recommendation.py)
client = df[df['code_client'] == 'CLT070730'].copy()
client['visit_date'] = pd.to_datetime(client['visit_date'])
client = client.sort_values('visit_date').drop_duplicates(subset=['code_article'], keep='last')

# Isoler le produit
prod = client[client['code_article'] == '25078RA3EABLACK4/128']
row = prod.iloc[0]

# Afficher les features
features_table = pd.DataFrame({
    'Feature': regressor_features,
    'Valeur': [row[f] for f in regressor_features],
    'Signification': [
        'Quantité moyenne par commande',
        'Écart-type des quantités',
        'Plus petite commande',
        'Plus grosse commande',
        'Quantité de la dernière commande',
        'Nombre total de commandes',
        'Jours depuis la dernière commande',
        'Délai moyen entre commandes (jours)',
        'Coefficient saisonnier du mois courant',
        'Coefficient saisonnier moyen du produit'
    ]
}).set_index('Feature')

features_table

### 3.2 Feature Importances — Ce que le régresseur regarde en priorité

In [ ]:
# Feature importances du modèle
importances = pd.Series(
    regressor.feature_importances_, 
    index=regressor_features
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#3b82f6' if imp < 0.15 else '#ef4444' for imp in importances]
bars = ax.barh(importances.index, importances.values, color=colors, edgecolor='white', height=0.6)

for bar, val in zip(bars, importances.values):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, 
            f'{val:.1%}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Feature Importances du Régresseur XGBoost', fontsize=14, fontweight='bold')

# Légende
red_patch = mpatches.Patch(color='#ef4444', label='Features dominantes (>15%)')
blue_patch = mpatches.Patch(color='#3b82f6', label='Features secondaires')
ax.legend(handles=[red_patch, blue_patch], loc='lower right')

ax.set_xlim(0, max(importances.values) + 0.06)
plt.tight_layout()
plt.show()

print("Le régresseur s'appuie principalement sur :")
print(f"  1. max_qty  (24.4%) — la plus grosse commande historique")
print(f"  2. avg_qty  (20.0%) — la moyenne historique")
print(f"  3. frequency (11.6%) — le nombre de commandes passées")

### 3.3 Pourquoi la prédiction est 192 et pas 146 (la moyenne) ?

In [ ]:
# Prédiction brute
X_input = prod[regressor_features].copy()
raw_pred = regressor.predict(X_input)[0]

print("═" * 65)
print("  DÉCOMPOSITION DE LA PRÉDICTION")
print("═" * 65)
print()
print(f"  Prédiction brute du régresseur : {raw_pred:.2f}")
print(f"  Après arrondi                  : {int(round(raw_pred))}")
print()
print("  ─── Pourquoi 192 et pas 146 (la moyenne) ? ───")
print()
print(f"  Le régresseur voit :")
print(f"    • max_qty  = 500  (feature #1, poids 24%)  → tire la prédiction VERS LE HAUT")
print(f"    • last_qty = 500  (feature #5, poids  9%)  → dernière commande = 500 → tire VERS LE HAUT")
print(f"    • avg_qty  = 146  (feature #2, poids 20%)  → ancre la prédiction autour de 146")
print(f"    • frequency = 11  (feature #3, poids 12%)  → client très fidèle → confiance")
print()
print(f"  ➡ Le régresseur fait un COMPROMIS entre :")
print(f"    - La moyenne historique (146) qui tire vers le bas")
print(f"    - La dernière commande (500) + max (500) qui tirent vers le haut")
print(f"    - Résultat : 192 ≈ moyenne pondérée vers le haut")
print()
print(f"  En d'autres termes : le modèle détecte une TENDANCE HAUSSIÈRE")
print(f"  (la dernière commande est bien au-dessus de la moyenne)")
print(f"  et ajuste sa prédiction à la hausse (+32% vs avg_qty)")
print("═" * 65)

In [ ]:
# Expérience : que se passe-t-il si on modifie last_qty ?
print("Expérience : effet de last_qty sur la prédiction")
print("─" * 55)

test_values = [9, 50, 100, 146, 200, 300, 500]
results = []

for val in test_values:
    X_test = X_input.copy()
    X_test['last_qty'] = val
    pred = regressor.predict(X_test)[0]
    results.append({'last_qty': val, 'prediction': round(pred, 1)})

exp_df = pd.DataFrame(results)
exp_df['delta_vs_avg'] = (exp_df['prediction'] - 146).round(1)
exp_df['delta_pct'] = ((exp_df['prediction'] / 146 - 1) * 100).round(1)
exp_df.columns = ['last_qty (modifié)', 'Prédiction', 'Δ vs moyenne', 'Δ %']
exp_df

---

## 4. Le Clamping : Comment la Prédiction Brute Devient la Suggestion Finale

In [ ]:
# Reproduire exactement _clamp_prediction() de recommendation.py
pred = raw_pred  # 192.25

hist_avg = float(row.get('avg_qty', pred))     # 145.9
hist_min = float(row.get('min_qty', 1.0))       # 9.0
hist_max = float(row.get('max_qty', pred * 2))   # 500.0

# Étape 1 : Bornes dures
lower = max(1, int(hist_min * 0.5))   # max(1, 4) = 4
upper = max(lower + 1, int(hist_max * 2.0))  # max(5, 1000) = 1000

# Étape 2 : Intervalle de confiance ±25%
qty_min = max(lower, int(pred * 0.75))   # max(4, 144) = 144
qty_max = min(upper, int(pred * 1.25))   # min(1000, 240) = 240

# Étape 3 : Garantie min < max
if qty_min >= qty_max:
    qty_max = qty_min + max(1, int(hist_avg * 0.25))

# Étape 4 : Suggestion clampée
qty_suggested = max(lower, min(int(round(pred)), upper))  # max(4, min(192, 1000)) = 192

print("═" * 65)
print("  PROCESSUS DE CLAMPING — Étape par Étape")
print("═" * 65)
print()
print(f"  Données historiques :")
print(f"    hist_min = {hist_min:.0f}  |  hist_max = {hist_max:.0f}  |  hist_avg = {hist_avg:.1f}")
print()
print(f"  Étape 1 — Bornes dures (garde-fous historiques) :")
print(f"    lower = max(1, {hist_min} × 0.5) = max(1, {int(hist_min*0.5)}) = {lower}")
print(f"    upper = max({lower+1}, {hist_max} × 2.0) = max({lower+1}, {int(hist_max*2)}) = {upper}")
print()
print(f"  Étape 2 — Intervalle de confiance (±25% de la prédiction) :")
print(f"    qty_min = max({lower}, {pred:.0f} × 0.75) = max({lower}, {int(pred*0.75)}) = {qty_min}")
print(f"    qty_max = min({upper}, {pred:.0f} × 1.25) = min({upper}, {int(pred*1.25)}) = {qty_max}")
print()
print(f"  Étape 3 — Suggestion finale :")
print(f"    qty_suggested = max({lower}, min({int(round(pred))}, {upper})) = {qty_suggested}")
print()
print(f"  ╔═══════════════════════════════════════════╗")
print(f"  ║  RÉSULTAT FINAL AFFICHÉ AU COMMERCIAL :   ║")
print(f"  ║                                           ║")
print(f"  ║  Quantité suggérée : {qty_suggested:>4}                 ║")
print(f"  ║  Quantité min      : {qty_min:>4}                 ║")
print(f"  ║  Quantité max      : {qty_max:>4}                 ║")
print(f"  ╚═══════════════════════════════════════════╝")
print("═" * 65)

In [ ]:
# Visualisation du clamping
fig, ax = plt.subplots(figsize=(14, 4))

# Axe des quantités
ax.set_xlim(0, 1100)
ax.set_ylim(-0.5, 3)

# Barre historique (min → max)
ax.barh(2, hist_max - hist_min, left=hist_min, height=0.4, color='#e0e7ff', edgecolor='#6366f1', label='Plage historique [9, 500]')

# Barre bornes dures (lower → upper)
ax.barh(1.3, upper - lower, left=lower, height=0.3, color='#fef3c7', edgecolor='#f59e0b', alpha=0.7, label=f'Bornes dures [{lower}, {upper}]')

# Barre intervalle de confiance
ax.barh(0.6, qty_max - qty_min, left=qty_min, height=0.5, color='#d1fae5', edgecolor='#10b981', linewidth=2, label=f'Intervalle confiance [{qty_min}, {qty_max}]')

# Points clés
ax.plot(hist_avg, 2, 'D', color='#ef4444', markersize=12, zorder=5, label=f'Moyenne historique ({hist_avg:.0f})')
ax.plot(qty_suggested, 0.6, '*', color='#22c55e', markersize=20, zorder=5, label=f'Prédiction régresseur ({qty_suggested})')
ax.plot(qtys[-1], 2, 's', color='#8b5cf6', markersize=10, zorder=5, label=f'Dernière commande ({int(qtys[-1])})')

# Commandes individuelles sur la ligne historique
for q in qtys:
    ax.plot(q, 2, '|', color='#6366f1', markersize=15, markeredgewidth=2)

ax.set_yticks([0.6, 1.3, 2])
ax.set_yticklabels(['Suggestion finale', 'Bornes dures', 'Historique client'], fontsize=11)
ax.set_xlabel('Quantité', fontsize=12)
ax.set_title('Clamping — Du brut à la suggestion finale', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

### 📌 Analyse du Clamping pour ce Cas

| Paramètre | Valeur | Commentaire |
|---|---|---|
| `lower` (borne basse) | **4** | `hist_min(9) × 0.5 = 4` → très bas car min_qty = 9 |
| `upper` (borne haute) | **1000** | `hist_max(500) × 2 = 1000` → très large car max_qty = 500 |
| `qty_min` | **144** | `pred(192) × 0.75 = 144` → le ±25% est la contrainte active |
| `qty_max` | **240** | `pred(192) × 1.25 = 240` → le ±25% est la contrainte active |
| `qty_suggested` | **192** | Non clampé car 4 ≤ 192 ≤ 1000 |

**Observation** : Les bornes historiques `[4, 1000]` sont très larges car ce client a un historique très variable (9 à 500). Le clamping ne corrige presque rien ici — c'est la prédiction brute du régresseur qui détermine le résultat.

---

## 5. ⚠️ Problème Critique : Le Régresseur est MOINS BON que la Baseline

C'est le point le plus important de ce notebook.

In [ ]:
# Métriques du modèle depuis les métadonnées
m_cont = metadata['metrics_continuous']
m_round = metadata['metrics_rounded']

print("═" * 65)
print("  MÉTRIQUES DU RÉGRESSEUR vs BASELINE (avg_qty)")
print("═" * 65)
print()
print(f"  Baseline : prédire simplement avg_qty (la moyenne historique)")
print(f"  Régresseur : XGBoost entraîné sur 10 features")
print()
print(f"  ┌───────────────────────────────────────────────────────────┐")
print(f"  │  Métrique    │  XGBoost    │  Baseline   │  Amélioration │")
print(f"  ├───────────────────────────────────────────────────────────┤")
print(f"  │  MAE         │  {m_round['mae']:>8.2f}   │  {m_round['mae_baseline']:>8.2f}   │  {m_round['improvement_mae_pct']:>+7.1f}%    │")
print(f"  │  RMSE        │  {m_round['rmse']:>8.2f}   │  {m_round['rmse_baseline']:>8.2f}   │  {m_round['improvement_rmse_pct']:>+7.1f}%    │")
print(f"  └───────────────────────────────────────────────────────────┘")
print()
print(f"  ⚠️  VERDICT : Le régresseur est {abs(m_round['improvement_mae_pct']):.1f}% PIRE que avg_qty !")
print()
print(f"  Concrètement :")
print(f"    - Prédire simplement la moyenne historique → erreur de {m_round['mae_baseline']:.1f} unités")
print(f"    - Utiliser le régresseur XGBoost           → erreur de {m_round['mae']:.1f} unités")
print(f"    - Le régresseur RAJOUTE {m_round['mae'] - m_round['mae_baseline']:.1f} unités d'erreur !")
print("═" * 65)

In [ ]:
# Comparaison visuelle
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE
ax = axes[0]
methods = ['Baseline\n(avg_qty)', 'XGBoost\nRégresseur']
mae_vals = [m_round['mae_baseline'], m_round['mae']]
colors = ['#22c55e', '#ef4444']
bars = ax.bar(methods, mae_vals, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, mae_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, 
            f'{val:.2f}', ha='center', va='bottom', fontsize=14, fontweight='bold')
ax.set_ylabel('MAE (unités)', fontsize=12)
ax.set_title('MAE — Erreur Absolue Moyenne', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(mae_vals) + 2)
ax.annotate(f'{abs(m_round["improvement_mae_pct"]):.1f}% PIRE', 
            xy=(1, m_round['mae']), fontsize=12, color='#ef4444', fontweight='bold',
            ha='center', va='bottom', xytext=(1, m_round['mae'] + 1.5))

# RMSE
ax = axes[1]
rmse_vals = [m_round['rmse_baseline'], m_round['rmse']]
bars = ax.bar(methods, rmse_vals, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, rmse_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{val:.2f}', ha='center', va='bottom', fontsize=14, fontweight='bold')
ax.set_ylabel('RMSE (unités)', fontsize=12)
ax.set_title('RMSE — Erreur Quadratique Moyenne', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(rmse_vals) + 5)
ax.annotate(f'{abs(m_round["improvement_rmse_pct"]):.1f}% PIRE', 
            xy=(1, m_round['rmse']), fontsize=12, color='#ef4444', fontweight='bold',
            ha='center', va='bottom', xytext=(1, m_round['rmse'] + 3))

plt.suptitle('Le régresseur XGBoost vs la baseline simple (avg_qty)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 📌 Pourquoi le Régresseur est Pire que avg_qty ?

Plusieurs raisons expliquent cette sous-performance :

**1. Variance extrême des quantités commandées**
- La médiane des `target_qty` dans le dataset est de **3 unités**, mais le P99 est de **239 unités**
- Le max atteint **4200 unités** — une distribution extrêmement asymétrique
- Le régresseur peine à capturer ces patterns très variables

**2. `avg_qty` est déjà la meilleure feature**
- `avg_qty` et `max_qty` représentent **44% de l'importance** du modèle
- En rajoutant d'autres features (recency, saisonnalité, etc.), le modèle **surfit** sur des signaux bruités
- Résultat : au lieu d'améliorer la prédiction, ces features rajoutent du bruit

**3. Nombre de données insuffisant pour l'hétérogénéité**
- Seulement **37 424 positifs** pour 356 clients très différents
- Chaque client a un comportement unique (grossiste vs détaillant vs occasionnel)
- Un seul modèle global ne peut pas bien capturer toutes ces dynamiques

**4. Pour notre cas spécifique (CLT070730)** :
- Écart-type (158) > Moyenne (146) → **Coefficient de variation = 108%**
- Le régresseur prédit 192 (tiré par `last_qty=500` et `max_qty=500`)
- Mais la vraie prochaine commande pourrait être 9 ou 500 — **impossible à prédire**

---

## 6. Distribution des Quantités dans le Dataset

In [ ]:
# Distribution des target_qty positives
positives = df[df['target_qty'] > 0]['target_qty']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution complète
ax = axes[0]
ax.hist(positives, bins=100, color='#3b82f6', alpha=0.8, edgecolor='white')
ax.axvline(x=positives.median(), color='#ef4444', linestyle='--', linewidth=2, label=f'Médiane = {positives.median():.0f}')
ax.axvline(x=192, color='#22c55e', linestyle='-', linewidth=2, label='Notre prédiction = 192')
ax.set_xlabel('target_qty', fontsize=12)
ax.set_ylabel('Fréquence', fontsize=12)
ax.set_title('Distribution de target_qty (tous positifs)', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xlim(0, 500)

# Zoom sur les petites quantités
ax = axes[1]
small = positives[positives <= 50]
ax.hist(small, bins=50, color='#8b5cf6', alpha=0.8, edgecolor='white')
ax.axvline(x=positives.median(), color='#ef4444', linestyle='--', linewidth=2, label=f'Médiane = {positives.median():.0f}')
ax.set_xlabel('target_qty', fontsize=12)
ax.set_ylabel('Fréquence', fontsize=12)
ax.set_title('Zoom : target_qty ≤ 50 (majorité des cas)', fontsize=13, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Statistiques target_qty (positifs uniquement) :")
print(f"  Total positifs : {len(positives)}")
print(f"  Médiane        : {positives.median():.0f} unités")
print(f"  Moyenne        : {positives.mean():.1f} unités")
print(f"  P90            : {positives.quantile(0.9):.0f} unités")
print(f"  P95            : {positives.quantile(0.95):.0f} unités")
print(f"  P99            : {positives.quantile(0.99):.0f} unités")
print(f"  Max            : {positives.max():.0f} unités")
print()
print(f"  ➡ {(positives <= 10).sum() / len(positives) * 100:.1f}% des commandes sont ≤ 10 unités")
print(f"  ➡ {(positives <= 50).sum() / len(positives) * 100:.1f}% des commandes sont ≤ 50 unités")
print(f"  ➡ Notre prédiction (192) est dans le top {(positives > 192).sum() / len(positives) * 100:.1f}%")

---

## 7. Verdict : La Prédiction de 192 est-elle Raisonnable ?

### ✅ Ce qui est correct

| Point | Explication |
|---|---|
| La prédiction est **dans la plage historique** | 192 est entre 9 (min) et 500 (max) |
| Le régresseur détecte une **tendance haussière** | La dernière commande (500) est bien au-dessus de la moyenne (146) |
| Le clamping fonctionne | Les bornes [144, 240] sont cohérentes avec ±25% |

### ❌ Ce qui pose problème

| Point | Explication |
|---|---|
| **Le régresseur est globalement pire que avg_qty** | MAE = 8.30 vs baseline 7.14 → il rajoute de l'erreur |
| **Variance trop élevée pour prédire** | CV=108% → commandes de 9 à 500, aucun pattern stable |
| **192 n'est PAS une quantité que le client commande réellement** | Ses commandes : 9, 10, 14, 22, 45, 50, 100, 255, 300, 300, 500 → jamais ~192 |
| **Le régresseur ne capture pas la bimodalité** | Ce client commande soit "petit" (~10-50) soit "gros" (~255-500), rarement entre les deux |

### 💡 Conclusion

La prédiction de **192** est techniquement dans une zone plausible, mais elle est **peu utile pour le commercial** car :
1. Elle ne correspond à aucun pattern réel du client
2. Le régresseur serait mieux remplacé par `avg_qty` (146) en l'état actuel
3. Pour ce type de client (très variable), un **intervalle** est plus utile qu'une prédiction ponctuelle

---

## 8. Pistes d'Amélioration du Régresseur

### Priorité 1 — Fallback vers avg_qty quand le régresseur n'est pas fiable

Si le coefficient de variation (CV = std_qty / avg_qty) est trop élevé, le régresseur ne peut pas faire mieux que la moyenne :

```python
# Dans recommendation.py
cv = std_qty / avg_qty if avg_qty > 0 else 999
if cv > 1.0:  # Variance trop haute
    # Utiliser avg_qty au lieu du régresseur
    raw_pred = avg_qty
    qty_source = "historique (variance trop élevée)"
```

### Priorité 2 — Utiliser la médiane au lieu de la moyenne

Pour des distributions asymétriques (comme ce client), la **médiane** est plus robuste que la moyenne :
- Moyenne de [9, 10, 14, 22, 45, 50, 100, 255, 300, 300, 500] = **146**
- Médiane = **50** (valeur centrale)

La médiane de 50 serait une suggestion plus utile pour le commercial.

### Priorité 3 — Ré-entraîner le régresseur avec des améliorations

1. **Log-transformer la target** : `y = log(target_qty)` pour réduire l'impact des outliers
2. **Ajouter la médiane comme feature** : `median_qty` est plus robuste que `avg_qty`
3. **Segmenter les clients** : entraîner des régresseurs séparés pour les grossistes vs détaillants
4. **Objectif MAE** : utiliser `reg:absoluteerror` au lieu de `reg:squarederror` pour être moins sensible aux outliers

In [ ]:
# Résumé final
print("╔" + "═" * 63 + "╗")
print("║" + " RÉSUMÉ — Notebook 12".center(63) + "║")
print("╠" + "═" * 63 + "╣")
print("║" + "".center(63) + "║")
print("║" + " Client : CLT070730".ljust(63) + "║")
print("║" + " Produit : REDMI 15C MIDNIGHT BLACK 4/128GB".ljust(63) + "║")
print("║" + " Prédiction régresseur : 192 unités".ljust(63) + "║")
print("║" + "".center(63) + "║")
print("║" + " Historique client : 11 commandes, de 9 à 500 unités".ljust(63) + "║")
print("║" + " Moyenne historique : 146 unités".ljust(63) + "║")
print("║" + " Dernière commande : 500 unités".ljust(63) + "║")
print("║" + "".center(63) + "║")
print("║" + " ⚠ Le régresseur est 16% PIRE que avg_qty (baseline)".ljust(63) + "║")
print("║" + " ⚠ CV = 108% → client trop imprévisible pour prédire".ljust(63) + "║")
print("║" + " ⚠ 192 ne correspond à aucune commande réelle".ljust(63) + "║")
print("║" + "".center(63) + "║")
print("║" + " → Recommandation : fallback vers avg_qty quand CV>1".ljust(63) + "║")
print("║" + " → Amélioration : log-transform + objectif MAE".ljust(63) + "║")
print("║" + "".center(63) + "║")
print("╚" + "═" * 63 + "╝")